# 09 — XGBoost Forecasting

Train a global gradient-boosted tree model and inspect validation performance, residuals and feature importance.

In [ ]:

from pathlib import Path
import os, sys
ROOT = Path.cwd()
while not (ROOT / "README.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("Project root:", ROOT)


In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from src.utils.config import load_yaml
from src.features.feature_pipeline import MODEL_FEATURES
from src.forecasting.xgboost_model import build_xgboost
from src.forecasting.evaluate import evaluate
df=pd.read_parquet(ROOT/"data/processed/forecast_features.parquet"); cfg=load_yaml("model_config.yaml"); maxd=df.date.max(); test_start=maxd-pd.Timedelta(days=cfg["test_days"]-1); val_start=test_start-pd.Timedelta(days=cfg["validation_days"]); train=df[df.date<val_start]; val=df[(df.date>=val_start)&(df.date<test_start)]
model=build_xgboost(cfg["xgboost"],cfg["random_seed"],cfg["n_jobs"]); model.fit(train[MODEL_FEATURES],train.demand); pred=model.predict(val[MODEL_FEATURES]).clip(min=0); print(evaluate(val.demand,pred))

In [ ]:
imp=pd.Series(model.feature_importances_,index=MODEL_FEATURES).sort_values(ascending=False).head(15); fig,ax=plt.subplots(figsize=(8,5)); imp.sort_values().plot(kind="barh",ax=ax); ax.set_title("XGBoost feature importance"); plt.show(); display(imp.to_frame("importance"))

In [ ]:
res=val.demand.to_numpy()-pred; fig,ax=plt.subplots(figsize=(8,4)); ax.hist(res,bins=40); ax.set_title("Validation residual distribution"); ax.set_xlabel("Actual - forecast"); plt.show()

### Modeling note

This model is global: one model learns shared patterns across all selected store-item series. That is computationally simpler and better suited to the portfolio-scale setting than fitting 30,490 independent models.